In [ ]:
import sys, subprocess

# Core dependencies — huggingface_hub must be force-reinstalled first
# to ensure XetAuthorizationError is available (required by transformers)
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--upgrade", "--force-reinstall", "huggingface_hub"
])

packages = [
    "torch",
    "transformers",
    "tokenizers",
    "einops",
    "addict",
    "easydict",
    "chromadb",
    "pypdfium2",
    "Pillow",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# flash-attn only makes sense on a CUDA machine — skip entirely on CPU
import torch
flash_attn_available = False
if torch.cuda.is_available():
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "wheel"])
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "flash-attn==2.7.3", "--no-build-isolation"
        ])
        flash_attn_available = True
    except Exception as e:
        print(f"flash-attn skipped: {e}")
else:
    print("No CUDA detected — skipping flash-attn.")

# Restart the kernel so all newly installed packages are loaded fresh
print("\nRestarting kernel to apply updated packages...")
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)


  Using cached huggingface_hub-1.24.0-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached filelock-3.32.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.2-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached packaging-26.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached tqdm-4.69.0-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached anyio-4.14.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.18-py3-none-any.whl.meta

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paddlex 3.7.2 requires PyYAML==6.0.2, but you have pyyaml 6.0.3 which is incompatible.
tokenizers 0.20.3 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.24.0 which is incompatible.
transformers 4.46.3 requires huggingface-hub<1.0,>=0.23.2, but you have huggingface-hub 1.24.0 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.24.0
    Uninstalling huggingface_hub-1.24.0:
      Successfully uninstalled huggingface_hub-1.24.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paddlex 3.7.2 requires PyYAML==6.0.2, but you have pyyaml 6.0.3 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


...


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


No CUDA detected — skipping flash-attn.

Restarting kernel to apply updated packages...


{'status': 'ok', 'restart': True}

: 

In [1]:
import os
import torch
from transformers import AutoModel, AutoTokenizer
import chromadb
from chromadb.utils import embedding_functions
import pypdfium2 as pdfium
from PIL import Image


/home/shivam/Documents/ACIP Template matching poc/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = 'deepseek-ai/DeepSeek-OCR-2'

# Auto-detect hardware
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
attn_impl = 'flash_attention_2' if (torch.cuda.is_available() and flash_attn_available) else 'eager'

print(f"Loading {MODEL_NAME} → device={device}, dtype={dtype}, attention={attn_impl}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    _attn_implementation=attn_impl,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=dtype,
).eval().to(device)

print("Model loaded successfully.")

# Initialize ChromaDB — recreate collection for a fresh run
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("document_templates")
except Exception:
    pass
collection = chroma_client.create_collection(
    name="document_templates",
    embedding_function=embedding_functions.DefaultEmbeddingFunction()
)
print("ChromaDB collection ready.")


Loading deepseek-ai/DeepSeek-OCR-2 → device=cpu, dtype=torch.float32, attention=eager


Encountered exception while importing torchvision: No module named 'torchvision'


ImportError: This modeling file requires the following packages that were not found in your environment: torchvision. Run `pip install torchvision`

In [ ]:
OCR_PROMPT = "<image>\nFree OCR. "

def ocr_page(image: Image.Image) -> str:
    """Run DeepSeek-OCR-2 on a PIL image and return the extracted text."""
    inputs = tokenizer(OCR_PROMPT, images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.batch_decode(output_ids[:, input_len:], skip_special_tokens=True)[0]


def get_document_fingerprints(file_path: str) -> list:
    """Generate text fingerprints for each page of a document (image or PDF)."""
    fingerprints = []
    base_name = os.path.basename(file_path)

    if file_path.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff', '.bmp', '.gif')):
        text = ocr_page(Image.open(file_path).convert("RGB"))
        fingerprints.append({
            "document": text,
            "metadata": {"template_name": base_name, "page_number": 1, "original_file": file_path}
        })
    elif file_path.lower().endswith('.pdf'):
        doc = pdfium.PdfDocument(file_path)
        for i, page_pdf in enumerate(doc):
            image = page_pdf.render(scale=2).to_pil()
            text = ocr_page(image)
            fingerprints.append({
                "document": text,
                "metadata": {
                    "template_name": f"{base_name}_page_{i+1}",
                    "page_number": i + 1,
                    "original_file": file_path
                }
            })
    else:
        print(f"Skipping unsupported file type: {file_path}")

    return fingerprints


### Embedding Templates into the Vector Store
Process all template files in the `templates/` directory.  
DeepSeek-OCR-2 extracts the text from each page, which is then stored as a vector fingerprint in ChromaDB.


In [ ]:
template_dir = 'templates/'

documents_to_add = []
metadatas_to_add = []
ids_to_add = []

for i, filename in enumerate(sorted(os.listdir(template_dir))):
    file_path = os.path.join(template_dir, filename)
    if not os.path.isfile(file_path):
        continue
    print(f"Processing template: {filename}")
    for j, fp_entry in enumerate(get_document_fingerprints(file_path)):
        documents_to_add.append(fp_entry["document"])
        metadatas_to_add.append(fp_entry["metadata"])
        ids_to_add.append(f"template_{i}_{j}")

if documents_to_add:
    collection.add(
        documents=documents_to_add,
        metadatas=metadatas_to_add,
        ids=ids_to_add,
    )
    print(f"\nAdded {len(documents_to_add)} fingerprint(s) to ChromaDB.")
else:
    print("No templates found in the templates/ directory.")


Processing template: CoO WCO (kyoto).png


/tmp/ipykernel_2571970/3783527725.py:4: DeprecationWarning: Please use `predict` instead.
  result = ocr.ocr(image_input)[0]


NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at /paddle/paddle/fluid/framework/new_executor/instruction/onednn/onednn_instruction.cc:116)


### Matching an Incoming Filled Document
Run DeepSeek-OCR-2 on the incoming document, then query ChromaDB to find the closest template match.


In [ ]:
incoming_doc_path = "test/certificat of origine.pdf"  # update path as needed
print(f"Processing incoming document: {os.path.basename(incoming_doc_path)}")

incoming_fingerprints = get_document_fingerprints(incoming_doc_path)

if not incoming_fingerprints:
    print(f"Could not generate fingerprints for: {incoming_doc_path}")
else:
    query_text = incoming_fingerprints[0]["document"]

    results = collection.query(
        query_texts=[query_text],
        n_results=1,
    )

    if results['ids'] and results['ids'][0]:
        meta     = results['metadatas'][0][0]
        distance = results['distances'][0][0]
        print(f"\nMatched Template : {meta['template_name']}")
        print(f"Original file    : {meta['original_file']}")
        print(f"Page             : {meta['page_number']}")
        print(f"Distance score   : {distance:.4f}")
    else:
        print("No matching template found.")


NameError: name 'os' is not defined